In [1]:
import hydra
from lmetk.data.datasets import Dataset_ETT_hour
from lmetk.data.utils import reconstruct_datetime_from_features

%load_ext autoreload
%autoreload 2


In [2]:
data = Dataset_ETT_hour(data_dir='datasets/ETT-small')

#/home/levtel/projects/TabPHM/datasets/ETT-small/ETTh1.csv

In [3]:
datasets = ['ETTh1']

for data_name in datasets:
#data_name = 'ETTh1'
    for flag in ['train', 'val', 'test']:
        with hydra.initialize(
                version_base="1.3",
                config_path='configs',
                job_name="run"
            ):
            #print('Current config file: ', config_file)
            parameters = hydra.compose(
                config_name="run.yaml",
                overrides=[f'dataset={data_name}', f'dataset.loader.flag={flag}'], 
                return_hydra_config=True, 
            )
            dataset_loader1 = hydra.utils.instantiate(parameters.dataset.loader)
            
    
            print(f"{flag} dataset size", dataset_loader1.__len__())

train dataset size 8161
val dataset size 2785
test dataset size 2785


### Behaviour of not refactored dataset

In [11]:
with hydra.initialize(
        version_base="1.3",
        config_path='configs',
        job_name="run"
    ):
    #print('Current config file: ', config_file)
    parameters = hydra.compose(
        config_name="run.yaml",
        overrides=[f'dataset=ETTh1', f'dataset.loader.flag=val'], 
        return_hydra_config=True, 
    )
    dataset_loader1 = hydra.utils.instantiate(parameters.dataset.loader)

for i in range(2783, 2787):
    seq_x, seq_y, seq_x_mark, seq_y_mark = dataset_loader1.__getitem__(i)
    print('Index', i)
    print(f"seq_x shape: {seq_x.shape}, seq_y shape: {seq_y.shape}, seq_x_mark shape: {seq_x_mark.shape}, seq_y_mark shape: {seq_y_mark.shape}")



Index 2783
seq_x shape: (384, 1), seq_y shape: (192, 1), seq_x_mark shape: (384, 4), seq_y_mark shape: (192, 4)
Index 2784
seq_x shape: (384, 1), seq_y shape: (192, 1), seq_x_mark shape: (384, 4), seq_y_mark shape: (192, 4)
Index 2785
seq_x shape: (384, 1), seq_y shape: (191, 1), seq_x_mark shape: (384, 4), seq_y_mark shape: (191, 4)
Index 2786
seq_x shape: (384, 1), seq_y shape: (190, 1), seq_x_mark shape: (384, 4), seq_y_mark shape: (190, 4)


### Refactored version

In [12]:
from lmetk.data.datasets import TimeSeriesDataset

data_name = 'ETTh1_refactored'

with hydra.initialize(
        version_base="1.3",
        config_path='configs',
        job_name="run"
    ):
    #print('Current config file: ', config_file)
    parameters = hydra.compose(
        config_name="run.yaml",
        overrides=[f'dataset={data_name}'], 
        return_hydra_config=True, 
    )
    dataset_loader2 = hydra.utils.instantiate(parameters.dataset.loader)


# Get params of splitting and sliding
data_splitter_params = parameters.dataset.loader.data_splitter
seq_len = data_splitter_params.seq_len
label_len = data_splitter_params.label_len
pred_len = data_splitter_params.pred_len

# get data
splitted_data, splitted_timestamps = dataset_loader2.get_data()

dataset_train = TimeSeriesDataset(data=splitted_data['train'], 
                  time_features=splitted_timestamps['train'], 
                  seq_len=seq_len, 
                  label_len=label_len, 
                  pred_len=pred_len)

dataset_val = TimeSeriesDataset(data=splitted_data['val'],
                    time_features=splitted_timestamps['val'], 
                    seq_len=seq_len, 
                    label_len=label_len, 
                    pred_len=pred_len)
dataset_test = TimeSeriesDataset(data=splitted_data['test'],
                    time_features=splitted_timestamps['test'], 
                    seq_len=seq_len, 
                    label_len=label_len, 
                    pred_len=pred_len)
print(f"Train dataset size: {dataset_train.__len__()}")
print(f"Validation dataset size: {dataset_val.__len__()}")
print(f"Test dataset size: {dataset_test.__len__()}")

Scaling features for train split using StandardScaler
Scaling features for val split using StandardScaler
Scaling features for test split using StandardScaler
Train dataset size: 8161
Validation dataset size: 2785
Test dataset size: 5805


In [13]:
for i in range(2783, 2787):
    seq_x, seq_y, seq_x_mark, seq_y_mark = dataset_val.__getitem__(i)
    print('Index', i)
    print(f"seq_x shape: {seq_x.shape}, seq_y shape: {seq_y.shape}, seq_x_mark shape: {seq_x_mark.shape}, seq_y_mark shape: {seq_y_mark.shape}")




Index 2783
seq_x shape: (384, 1), seq_y shape: (192, 1), seq_x_mark shape: (384, 4), seq_y_mark shape: (192, 4)
Index 2784
seq_x shape: (384, 1), seq_y shape: (192, 1), seq_x_mark shape: (384, 4), seq_y_mark shape: (192, 4)


AssertionError: Target sequence indices out of bounds

### Checking if data is the same 

In [14]:
for i in range(5):
    seq_x1, seq_y1, seq_x_mark1, seq_y_mark1 = dataset_val.__getitem__(i)  
    seq_x2, seq_y2, seq_x_mark2, seq_y_mark2 = dataset_loader1.__getitem__(i)

    # Check if the data is the same
    assert (seq_x1 == seq_x2).all(), f"seq_x1 and seq_x2 are not the same at index {i}"
    assert (seq_y1 == seq_y2).all(), f"seq_y1 and seq_y2 are not the same at index {i}"
    assert (seq_x_mark1 == seq_x_mark2).all(), f"seq_x_mark1 and seq_x_mark2 are not the same at index {i}"
    assert (seq_y_mark1 == seq_y_mark2).all(), f"seq_y_mark1 and seq_y_mark2 are not the same at index {i}"
    print(f"Data at index {i} is the same in both datasets.")

# NOTE THAT ETTH does not use the normalized timestamps

Data at index 0 is the same in both datasets.
Data at index 1 is the same in both datasets.
Data at index 2 is the same in both datasets.
Data at index 3 is the same in both datasets.
Data at index 4 is the same in both datasets.


### Railway: classical dataset 

In [16]:
import hydra
data_name = 'railway'

with hydra.initialize(
        version_base="1.3",
        config_path='configs',
        job_name="run"
    ):
    #print('Current config file: ', config_file)
    parameters = hydra.compose(
        config_name="run.yaml",
        overrides=[f'dataset={data_name}'], 
        return_hydra_config=True, 
    )
    dataset_loader2 = hydra.utils.instantiate(parameters.dataset.loader)


In [17]:
from lmetk.data.datasets import TimeSeriesDataset
data_splitter_params = parameters.dataset.loader.data_splitter
seq_len = data_splitter_params.seq_len
label_len = data_splitter_params.label_len
pred_len = data_splitter_params.pred_len

# get data
splitted_data, splitted_timestamps = dataset_loader2.get_data()

dataset_train = TimeSeriesDataset(data=splitted_data['train'], 
                  time_features=splitted_timestamps['train'], 
                  seq_len=seq_len, 
                  label_len=label_len, 
                  pred_len=pred_len)

dataset_val = TimeSeriesDataset(data=splitted_data['val'],
                    time_features=splitted_timestamps['val'], 
                    seq_len=seq_len, 
                    label_len=label_len, 
                    pred_len=pred_len)
dataset_test = TimeSeriesDataset(data=splitted_data['test'],
                    time_features=splitted_timestamps['test'], 
                    seq_len=seq_len, 
                    label_len=label_len, 
                    pred_len=pred_len)
print(f"Train dataset size: {dataset_train.__len__()}")
print(f"Validation dataset size: {dataset_val.__len__()}")
print(f"Test dataset size: {dataset_test.__len__()}")

Train dataset size: 37921
Validation dataset size: 4201
Test dataset size: 8185


In [18]:
# Test should stard from "2023-01-24 00:00:00" (if apply_lookback == False then __getitem__(seq_len) otherwise __getitem__(0))
seq_x, seq_y, seq_x_mark, seq_y_mark = dataset_test.__getitem__(seq_len)
reconstruct_datetime_from_features(seq_x_mark, ["year", "month", "day", "weekday", "hour"])

0    2023-01-24 00:00:00
1    2023-01-24 01:00:00
2    2023-01-24 02:00:00
3    2023-01-24 03:00:00
4    2023-01-24 04:00:00
5    2023-01-24 05:00:00
6    2023-01-24 06:00:00
7    2023-01-24 07:00:00
8    2023-01-24 08:00:00
9    2023-01-24 09:00:00
10   2023-01-24 10:00:00
11   2023-01-24 11:00:00
12   2023-01-24 12:00:00
13   2023-01-24 13:00:00
14   2023-01-24 14:00:00
15   2023-01-24 15:00:00
16   2023-01-24 16:00:00
17   2023-01-24 17:00:00
18   2023-01-24 18:00:00
19   2023-01-24 19:00:00
20   2023-01-24 20:00:00
21   2023-01-24 21:00:00
22   2023-01-24 22:00:00
23   2023-01-24 23:00:00
24   2023-01-25 00:00:00
25   2023-01-25 01:00:00
26   2023-01-25 02:00:00
27   2023-01-25 03:00:00
28   2023-01-25 04:00:00
29   2023-01-25 05:00:00
30   2023-01-25 06:00:00
31   2023-01-25 07:00:00
32   2023-01-25 08:00:00
33   2023-01-25 09:00:00
34   2023-01-25 10:00:00
35   2023-01-25 11:00:00
36   2023-01-25 12:00:00
37   2023-01-25 13:00:00
38   2023-01-25 14:00:00
39   2023-01-25 15:00:00


In [19]:
seq_x, seq_y, seq_x_mark, seq_y_mark = dataset_val.__getitem__(seq_len)
reconstruct_datetime_from_features(seq_x_mark, ["year", "month", "day", "weekday", "hour"])

0    2022-08-02 00:00:00
1    2022-08-02 01:00:00
2    2022-08-02 02:00:00
3    2022-08-02 03:00:00
4    2022-08-02 04:00:00
5    2022-08-02 05:00:00
6    2022-08-02 06:00:00
7    2022-08-02 07:00:00
8    2022-08-02 08:00:00
9    2022-08-02 09:00:00
10   2022-08-02 10:00:00
11   2022-08-02 11:00:00
12   2022-08-02 12:00:00
13   2022-08-02 13:00:00
14   2022-08-02 14:00:00
15   2022-08-02 15:00:00
16   2022-08-02 16:00:00
17   2022-08-02 17:00:00
18   2022-08-02 18:00:00
19   2022-08-02 19:00:00
20   2022-08-02 20:00:00
21   2022-08-02 21:00:00
22   2022-08-02 22:00:00
23   2022-08-02 23:00:00
24   2022-08-03 00:00:00
25   2022-08-03 01:00:00
26   2022-08-03 02:00:00
27   2022-08-03 03:00:00
28   2022-08-03 04:00:00
29   2022-08-03 05:00:00
30   2022-08-03 06:00:00
31   2022-08-03 07:00:00
32   2022-08-03 08:00:00
33   2022-08-03 09:00:00
34   2022-08-03 10:00:00
35   2022-08-03 11:00:00
36   2022-08-03 12:00:00
37   2022-08-03 13:00:00
38   2022-08-03 14:00:00
39   2022-08-03 15:00:00


### Railway efficient dataset

In [22]:
import hydra
%load_ext autoreload
%autoreload 2
data_name = 'railway'

with hydra.initialize(
        version_base="1.3",
        config_path='configs',
        job_name="run"
    ):
    #print('Current config file: ', config_file)
    parameters = hydra.compose(
        config_name="run.yaml",
        overrides=[f'dataset={data_name}'], 
        return_hydra_config=True, 
    )
    dataset_loader2 = hydra.utils.instantiate(parameters.dataset.loader)

data_splitter_params = parameters.dataset.loader.data_splitter
seq_len = data_splitter_params.seq_len
label_len = data_splitter_params.label_len
pred_len = data_splitter_params.pred_len

# get data
splitted_data, splitted_timestamps = dataset_loader2.get_data()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [23]:
parameters.dataset.loader

{'_target_': 'lmetk.data.datasets.RailwayLoader', 'load_arguments': {'data_name': 'railway', 'data_folder': 'railway', 'data_path': '${paths.data_dir}/${dataset.loader.load_arguments.data_folder}/${dataset.loader.load_arguments.data_name}.csv', 'samples_per_day': 24, 'target_name': 'Bahnlast', 'timestamp_name': 'datetime', 'trend_decomp': False, 'trend_decomp_rolling_window': None, 'time_features': ['year', 'month', 'day', 'weekday', 'hour'], 'normalize_time_features': False}, 'data_splitter': {'_target_': 'lmetk.data.preprocessing.splits.DateBasedSplitter', 'test_start': '2023-01-24 00:00:00', 'train_ratio': 0.9, 'val_ratio': 0.1, 'seq_len': 48, 'label_len': 24, 'pred_len': 24, 'apply_lookback': True}, 'scaler': {'features': {'_target_': 'sklearn.preprocessing.MinMaxScaler'}}}

In [39]:
from lmetk.data.datasets.context_dataset import ContextBatchDataset #,ContextBatchDataset2
for mode in ['train', 'val', 'test']:
    
    context_dataset = ContextBatchDataset(
        features=splitted_data[mode], 
        time_features=splitted_timestamps[mode], 
        seq_len=seq_len, 
        label_len=label_len, 
        pred_len=pred_len,
        mask=dataset_loader2.data_dict['mask'],
        stride=1,
    )
    print(f"ContextBatchDataset {mode} size: {context_dataset.__len__()}")
    for i in range(context_dataset.__len__()):
        context_dataset.__getitem__(i)
    d = context_dataset.__getitem__(i)

    for key in d['target'].keys():
        # pring shape of each key
        try:
            print(f"{key}: {d['target'][key].shape}")
        except:
            pass
    if mode == 'test':
        # Test should stard from "2023-01-24 00:00:00"
        d = context_dataset.__getitem__(seq_len)
        print(reconstruct_datetime_from_features(d['target']['x_time'].long().numpy(), ["year", "month", "day", "weekday", "hour"])[:2])
        


ContextBatchDataset train size: 37921
x: torch.Size([48, 1])
y: torch.Size([48, 1])
x_time: torch.Size([48, 5])
y_time: torch.Size([48, 5])
ContextBatchDataset val size: 4201
x: torch.Size([48, 1])
y: torch.Size([48, 1])
x_time: torch.Size([48, 5])
y_time: torch.Size([48, 5])
ContextBatchDataset test size: 8185
x: torch.Size([48, 1])
y: torch.Size([48, 1])
x_time: torch.Size([48, 5])
y_time: torch.Size([48, 5])
0   2023-01-24 00:00:00
1   2023-01-24 01:00:00
dtype: datetime64[ns]


tensor([37944, 37992])

In [ ]:
from lmetk.data.datasets.railway_traction import SpatialDataModule
dm = SpatialDataModule(ts_in=24, batch_size=10)
dm.setup()
batch = next(iter(dm.train_dataloader()))